In [ ]:
from sklearn.cross_decomposition import PLSRegression
from sklearn.compose import TransformedTargetRegressor
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import make_scorer
from sklearn.model_selection import GridSearchCV, LeaveOneGroupOut, cross_val_predict
from sklearn.callback import ProgressBar
from sklearn import set_config
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import scipy
import os

if os.getcwd().endswith("notebooks"):
    os.chdir("..")

from src.model import CircularTransformer, jammalamadaka_sharma_corr, angular_absolute_error

set_config(enable_metadata_routing=True)

### Read data

In [ ]:
climate = pd.read_parquet("phenocam_site_in_usa_climate.parquet")
climate.columns = ["year", "yday", "prcp", "tmax", "tmin", "vp", "site"]
pheno   = pd.read_parquet("phenocam_phenology_model.parquet")
site_meta = pd.read_parquet("phenocam_site_in_usa.parquet")

### Summarize meteorology

We construct precipitation and temperature time series at daily and monthly temporal resolution. 

In [ ]:
climate_monthly = climate\
    .assign(month = lambda x: "month" + np.minimum(12, ((x.yday // 30)+1)).astype(str))\
    .groupby(["site", "year", "month"])[["prcp", "tmax"]]\
    .agg("mean")\
    .reset_index()\
    .pivot(index=["site", "year"], columns="month")

climate_monthly.columns = ['_'.join(col).strip() for col in climate_monthly.columns.values]

climate_monthly.head()

In [ ]:
# Convert climate to wide format - essentially make the full prcp/tmax time series
# the columns
climate_daily = climate\
    .assign(yday=lambda x: "yday" + x.yday.astype(str))\
    .drop(columns=["tmin", "vp"])\
    .pivot(index=["site", "year"], columns="yday")
# Clean column names
climate_daily.columns = ['_'.join(col).strip() for col in climate_daily.columns.values]
climate_daily.head()

### Join phenological transitions

In [ ]:
# Join phenological transitions onto site-years
climate_pheno_monthly = pd.merge(
    climate_monthly,
    pheno[["site", "transition_year_rising", "veg_type", "doy_transition_50_rising"]],
    how="inner",
    left_on=["site", "year"],
    right_on=["site", "transition_year_rising"]
)

climate_pheno_daily = pd.merge(
    climate_daily,
    pheno[["site", "transition_year_rising", "veg_type", "doy_transition_50_rising"]],
    how="inner",
    left_on=["site", "year"],
    right_on=["site", "transition_year_rising"]
)

### PLSR model

Structure this as a transformed target regressor to account for angular response. For scoring, use J-S correlation coefficient to account for wrapping.

In [ ]:
js_scorer = make_scorer(jammalamadaka_sharma_corr)
aae_scorer = make_scorer(lambda x, y: angular_absolute_error(x, y).mean(), greater_is_better=False)

### Grid-search for N components

In [ ]:
import warnings
warnings.filterwarnings(
    "ignore", message=".*The estimator TransformedTargetRegressor does not support callbacks.*"
)

In [ ]:
cv_daily = GridSearchCV(
    TransformedTargetRegressor(
        regressor=PLSRegression(),
        transformer=CircularTransformer(cycle_length=365),
        check_inverse=False
    ),
    param_grid = dict(regressor__n_components = list(range(1, 10))),
    scoring = dict(js_corr=js_scorer, aae=aae_scorer),
    refit="js_corr"
)

cv_daily.set_callbacks(ProgressBar())

X_daily = climate_pheno_daily.filter(regex="(tmax|prcp)")
y_daily = climate_pheno_daily["doy_transition_50_rising"]

cv_daily.fit(X_daily, y_daily)

In [ ]:
print("Best parameters")
print(cv_daily.best_params_)

print("JS Score on full dataset")
print(js_scorer(cv_daily, X_daily, y_daily))

print("AAE on full dataset")
print(-aae_scorer(cv_daily, X_daily, y_daily))

Repeat at monthly timestep. Hold components at 4 for comparability.

In [ ]:
cv_monthly = GridSearchCV(
    TransformedTargetRegressor(
        regressor=PLSRegression(),
        transformer=CircularTransformer(cycle_length=365),
        check_inverse=False
    ),
    param_grid = dict(regressor__n_components = [4]),
    scoring = dict(js_corr=js_scorer, aae=aae_scorer),
    refit = "js_corr"
)

cv_monthly.set_callbacks(ProgressBar())

X_monthly = climate_pheno_monthly.filter(regex="(tmax|prcp)")
y_monthly = climate_pheno_monthly["doy_transition_50_rising"]

cv_monthly.fit(X_monthly, y_monthly)

In [ ]:
print("Best parameters")
print(cv_monthly.best_params_)

print("JS Score on full dataset")
print(js_scorer(cv_monthly, X_monthly, y_monthly))

print("AAE on full dataset")
print(-aae_scorer(cv_monthly, X_monthly, y_monthly))

### Generate out-of-sample predictions

In [ ]:
climate_pheno_daily["model_doy_transition_50_rising"] = cross_val_predict(
    cv_daily.estimator, 
    X_daily, 
    y_daily, 
    cv=LeaveOneGroupOut(), 
    params=dict(groups=climate_pheno_daily["site"])
)

climate_pheno_monthly["model_doy_transition_50_rising"] = cross_val_predict(
    cv_monthly.estimator, 
    X_monthly, 
    y_monthly, 
    cv=LeaveOneGroupOut(), 
    params=dict(groups=climate_pheno_monthly["site"])
)

### Aggregate MAE

In [ ]:
climate_pheno_daily["model_error_doy"] = angular_absolute_error(
    climate_pheno_daily["doy_transition_50_rising"],
    climate_pheno_daily["model_doy_transition_50_rising"]
)

climate_pheno_monthly["model_error_doy"] = angular_absolute_error(
    climate_pheno_monthly["doy_transition_50_rising"],
    climate_pheno_monthly["model_doy_transition_50_rising"]
)

In [ ]:
daily_error_agg = climate_pheno_daily\
    .groupby("site")\
    .agg(dict(model_error_doy=["mean", "count"]))\
    .droplevel(0, axis=1)\
    .join(site_meta[["MAT_daymet", "MAP_daymet"]])

monthly_error_agg = climate_pheno_monthly\
    .groupby("site")\
    .agg(dict(model_error_doy=["mean", "count"]))\
    .droplevel(0, axis=1)\
    .join(site_meta[["MAT_daymet", "MAP_daymet"]])

Within-site average error over climate space.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, sharey=True, sharex=False, figsize=(8, 4))

size_gain = 4
monthly_error_agg["count_gain"] = monthly_error_agg["count"] * size_gain
daily_error_agg["count_gain"] = daily_error_agg["count"] * size_gain

plot_args = dict(
    kind="scatter",
    x="MAT_daymet", 
    y="MAP_daymet", 
    c="mean", 
    s="count_gain",
    edgecolors="black",
    vmin=0, 
    vmax=90, 
    colorbar=False,
    linewidth=0.5
)

daily_error_agg.plot(
    ax=ax1,
    **plot_args
)

monthly_error_agg.plot(
    ax=ax2,
    **plot_args
)

for ax in (ax1, ax2):
    ax.grid()
    ax.set_xlabel("")
    ax.set_ylabel("")

cbar = plt.colorbar(ax1.collections[0], ax=(ax1, ax2), orientation='vertical', fraction=.1)
cbar.set_label("Average site error (days)")

ax1.set_title("Daily timesteps")
ax2.set_title("Monthly timesteps")

fig.supxlabel("MAT (deg C)")
fig.supylabel("MAP (mm)")

In [ ]:
pd.merge(
    daily_error_agg,
    monthly_error_agg,
    left_index=True,
    right_index=True,
    suffixes=["_daily", "_monthly"]
).plot(x="mean_daily", y="mean_monthly", kind="scatter", s="count_gain_daily", edgecolor="black", linewidth=0.5)

plt.grid()
plt.xlabel("Daily timestep MAE")
plt.ylabel("Monthly timestep MAE")
plt.axline((0, 0), (1, 1), color="black")

### Distinct models by veg type

Now that we know the appropriate time step and number of components to use, we can see if there is much of a benefit to training distinct PLSR models on each vegetation type. This might improve the greater error in hot/dry areas seen above.

In [ ]:
climate_pheno_monthly.groupby("veg_type")["site"].agg("count")

In [ ]:
MIN_OBS = 100
ESTIMATOR_ARGS = dict(
    n_components=4
)

estimators = {}

cv_predict_results = []

for veg_type, sub_df in climate_pheno_monthly.groupby("veg_type"):
    if sub_df.shape[0] < MIN_OBS:
        print(f"{veg_type} has fewer than {MIN_OBS} observations. Fitting global model.")
        model_df = climate_pheno_monthly.copy()
    else:
        model_df = sub_df.copy()

    X_sub = model_df.filter(regex="(tmax|prcp)")
    y_sub = model_df["doy_transition_50_rising"]
    
    # Fit estimator
    estimator = TransformedTargetRegressor(
        regressor=PLSRegression(**ESTIMATOR_ARGS),
        transformer=CircularTransformer(cycle_length=365),
        check_inverse=False
    ).fit(X_sub, y_sub)
    estimators[veg_type] = estimator
    
    # Run LOO predictions by site    
    y_sub_hat = cross_val_predict(
        estimator,
        X_sub,
        y_sub,
        cv=LeaveOneGroupOut(),
        params=dict(groups=model_df["site"])
    )

    model_df["model_cv_doy_transition_50_rising"] = y_sub_hat

    cv_predict_results.append(model_df[model_df.veg_type == veg_type])

Global performance metrics.

In [ ]:
cv_predict = pd.concat(cv_predict_results, axis=0)
cv_predict["model_error_doy"] = angular_absolute_error(cv_predict["doy_transition_50_rising"], cv_predict["model_cv_doy_transition_50_rising"])

print(f"Global mean angular absolute error: {cv_predict["model_error_doy"].mean():.2f}")
print(f"Global JS correlation: {jammalamadaka_sharma_corr(cv_predict["doy_transition_50_rising"], cv_predict["model_cv_doy_transition_50_rising"]):.2f}")

In [ ]:
monthly_cv_error_agg = cv_predict\
    .groupby("site")\
    .agg(dict(model_error_doy=["mean", "count"]))\
    .droplevel(0, axis=1)\
    .join(site_meta[["MAT_daymet", "MAP_daymet"]])

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, sharey=True, sharex=False, figsize=(8, 4))

size_gain = 4
monthly_error_agg["count_gain"] = monthly_error_agg["count"] * size_gain
monthly_cv_error_agg["count_gain"] = daily_error_agg["count"] * size_gain

plot_args = dict(
    kind="scatter",
    x="MAT_daymet", 
    y="MAP_daymet", 
    c="mean", 
    s="count_gain",
    edgecolors="black",
    vmin=0, 
    vmax=90, 
    colorbar=False,
    linewidth=0.5
)

monthly_cv_error_agg.plot(
    ax=ax1,
    **plot_args
)

monthly_error_agg.plot(
    ax=ax2,
    **plot_args
)

for ax in (ax1, ax2):
    ax.grid()
    ax.set_xlabel("")
    ax.set_ylabel("")

cbar = plt.colorbar(ax1.collections[0], ax=(ax1, ax2), orientation='vertical', fraction=.1)
cbar.set_label("Average site error (days)")

ax1.set_title("Separate models by veg type")
ax2.set_title("Single model")

fig.supxlabel("MAT (deg C)")
fig.supylabel("MAP (mm)")

In [ ]:
pd.merge(
    monthly_cv_error_agg,
    monthly_error_agg,
    left_index=True,
    right_index=True,
    suffixes=["_cv_monthly", "_monthly"]
).plot(x="mean_cv_monthly", y="mean_monthly", kind="scatter", s="count_gain_monthly", edgecolor="black", linewidth=0.5)

plt.grid()
plt.xlabel("Multiple model MAE")
plt.ylabel("Single model MAE")
plt.axline((0, 0), (1, 1), color="black")

There doesn't seem to be evidence in favor of separate models for each vegetation type.

### Refine OLS fit with MLE

An angular transformation helps us fit reasonable models using the sklearn API. But, based on results with the forcing model, we might do better by using maximum likelihood estimation.

In [ ]:
# Verify that we can manually reproduce predictions
regr_coef = cv_monthly.best_estimator_.regressor_.coef_
regr_mean = cv_monthly.best_estimator_.regressor_._x_mean
regr_inpt = cv_monthly.best_estimator_.regressor_.intercept_
transformer = cv_monthly.best_estimator_.transformer_

y_hat_actual = cv_monthly.best_estimator_.predict(X_monthly).squeeze()
y_hat_manual = transformer.inverse_transform((X_monthly - regr_mean) @ regr_coef.T + regr_inpt).squeeze()

assert(np.allclose(y_hat_actual, y_hat_manual))

In [ ]:
# Determine dispersion parameter for the VM distribution
vm_args = scipy.stats.fit(
    scipy.stats.vonmises, 
    transformer.to_radians(climate_pheno_monthly["doy_transition_50_rising"]),
    bounds=dict(
        kappa=(0, 10),
        loc=(-np.pi, np.pi)
    ),
    guess=dict(loc=transformer.to_radians(climate_pheno_monthly["doy_transition_50_rising"]).mean())
)

vm_args.plot()

In [ ]:
# Set up likelihood function
y_actual_radians = transformer.to_radians(climate_pheno_monthly["doy_transition_50_rising"])

def von_mises_log_likelihood(params):
    y_hat_radians = ((X_monthly - regr_mean) @ params + regr_inpt).squeeze()
    logpdf = scipy.stats.vonmises.logpdf(y_actual_radians, kappa=vm_args.params.kappa, loc=y_hat_radians)
    return -np.sum(logpdf)

# Make sure it works
von_mises_log_likelihood(regr_coef.T)

In [ ]:
# Optimize!
# The initial coefficients should be close to the MLE optimum. They are all
# near zero. Prevent massive model change by bounding to +/- the abs max of
# the initial fit.
coef_lower_bound = (np.zeros_like(regr_coef) - np.abs(np.max(regr_coef))).squeeze()
coef_upper_bound = (np.zeros_like(regr_coef) + np.abs(np.max(regr_coef))).squeeze()

global_result = scipy.optimize.minimize(
    lambda x: von_mises_log_likelihood(x) / 100,
    regr_coef.T.squeeze(),
    method="L-BFGS-B",
    bounds=np.stack((coef_lower_bound, coef_upper_bound), axis=1)
)

In [ ]:
y_hat_mle = transformer.inverse_transform((X_monthly - regr_mean) @ global_result.x[:, np.newaxis] + regr_inpt).squeeze()
print(jammalamadaka_sharma_corr(y_monthly, y_hat_mle, cycle_length=365))
print(angular_absolute_error(y_monthly, y_hat_mle, cycle_length=365).mean())

Doesn't seem to improve predictions :/

### Save final model and persist

In [ ]:
final_plsr_model = TransformedTargetRegressor(
    regressor=PLSRegression(n_components=4),
    transformer=CircularTransformer(cycle_length=365),
    check_inverse=False
).fit(X_monthly, y_monthly)

In [ ]:
import skops.io as skops

skops.dump(final_plsr_model, "phenocam_monthly_plsr.skops")

In [ ]:
final_plsr_model.regressor_.feature_names_in_

In [ ]:
unknown_types = skops.get_untrusted_types(file="phenocam_monthly_plsr.skops")
print(unknown_types)

In [ ]:
model_reload = skops.load("phenocam_monthly_plsr.skops", trusted=unknown_types)

In [ ]:
assert np.allclose(model_reload.predict(X_monthly), final_plsr_model.predict(X_monthly))